# Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import ADASYN
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor

: 

In [ ]:
employee= pd.read_csv(r"/kaggle/input/datasets/prahazra/employee-attirtion-data/Palo Alto Networks.csv")
employee.sample(5)

In [ ]:
employee.isnull().sum()

In [ ]:
employee.duplicated().sum()

In [ ]:
employee.describe()

In [ ]:
num_cols= employee.select_dtypes(include=['int64', 'float64']).columns.tolist()
Q1 = employee[num_cols].quantile(0.25)
Q3 = employee[num_cols].quantile(0.75)
IQR = Q3 - Q1
outliers_mask = (employee[num_cols] < (Q1 - 1.5 * IQR)) | (employee[num_cols] > (Q3 + 1.5 * IQR))

print("--- Outlier Count Per Column ---")
print(outliers_mask.sum())
print("\nTotal rows with at least one outlier:", outliers_mask.any(axis=1).sum())

In [ ]:
employee.dtypes

In [ ]:
features_to_be_float= ['DailyRate', 'HourlyRate', 'MonthlyRate', 'MonthlyIncome', 'PercentSalaryHike']
employee[features_to_be_float] = employee[features_to_be_float].astype(float)

In [ ]:
print(f"NumCompaniesWorked{employee['NumCompaniesWorked'].unique()}")
print(f"TotalWorkingYears{employee['TotalWorkingYears'].unique()}")
print(f"YearsInCurrentRole{employee['YearsInCurrentRole'].unique()}")

# Feature Creation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def create_features(df):
    # FIXED: Ensured we work with a copy to prevent SettingWithCopyWarning
    df = df.copy()
    
    # FIXED: Replaced 'employee' with 'df' to avoid breaking on local variables
    satisfaction_cols = ['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']
    correlations = df[satisfaction_cols].corrwith(df['Attrition']).abs()
    weights = correlations / correlations.sum()
    
    df['OverallSatisfaction'] = np.ceil(
        (weights['JobSatisfaction'] * df['JobSatisfaction']) +
        (weights['EnvironmentSatisfaction'] * df['EnvironmentSatisfaction']) +
        (weights['RelationshipSatisfaction'] * df['RelationshipSatisfaction']) +
        (weights['WorkLifeBalance'] * df['WorkLifeBalance'])
    ).astype(int)
    
    df['CompensationIncomeDifference'] = df['MonthlyIncome'] - df['MonthlyRate']
    
    # ------- Used Data Statistics Here ------------------------------------------------------
    role_level_avg = df.groupby(['JobRole', 'JobLevel'])['MonthlyIncome'].transform('mean')
    scaler = StandardScaler()
    cols = ['JobInvolvement', 'PerformanceRating', 'JobLevel']
    scaled_values = scaler.fit_transform(df[cols])
    
    # FIXED: Aligned the index of df_scaled to df.index to prevent NaN values
    df_scaled = pd.DataFrame(scaled_values, columns=cols, index=df.index)
    # ----------------------------------------------------------------------------------------
    
    df['CompensationRatio'] = df['MonthlyIncome'] / role_level_avg
    df['IncomeExperienceRatio'] = df['MonthlyIncome'] / (df['TotalWorkingYears'] + 1)
    df['PromotionDelay'] = df['YearsSinceLastPromotion'] / (df['YearsInCurrentRole'] + 1)
    df['EngagementCompositeScore'] = df_scaled.sum(axis=1)
    
    # FIXED: Added parentheses to fix operator precedence and changed text to match 'Travel Frequently'
    df['IfStressed'] = np.where(
        (df['OverTime'] == 'Yes') & (df['BusinessTravel'] == 'Travel Frequently'), 1, 0
    )
    
    # FIXED: Fixed spelling of 'YearsWithCurrManager' and corrected the np.where parenthesis placement
    df['ManagerFriction'] = np.where(
        (df['YearsWithCurrManager'] < 1) & (df['YearsAtCompany'] > 3), 1, 0
    )
    
    # FIXED: Fixed spelling of 'PercentSalaryHike' and 'PerformanceRating'
    df['CompensationDiscrepancy'] = df['PercentSalaryHike'] / df['PerformanceRating']
    
    df['JobChangeFrequency'] = df['TotalWorkingYears'] / (df['NumCompaniesWorked'] + 1)
    df['LoyalityScore'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1)
    df['RoleStagnationIndex'] = df['YearsInCurrentRole'] / (df['YearsAtCompany'] + 1e-5)
    
    # FIXED: Cast the boolean result explicitly to an integer (0 or 1) as implied by its design
    df['ExternalExperienceValue'] = (df['TotalWorkingYears'] - df['YearsAtCompany']) / df['JobLevel']
    df['EducationIncomeRatio'] = df['MonthlyIncome'] / df.groupby('Education')['MonthlyIncome'].transform('median')
    df['CommuteBurnout'] = ((df['OverTime'] == 'Yes') & (df['DistanceFromHome'] > df['DistanceFromHome'].quantile(0.66))).astype('int')

    return df


In [ ]:
employee_featured = create_features(employee)
employee.columns

In [ ]:
# 1. Isolate numeric columns (excluding the target column 'Attrition')
vif_df = employee_featured.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).copy()
if 'Attrition' in vif_df.columns:
    vif_df = vif_df.drop(columns=['Attrition'])

# 2. Safety Cleanup (Pre-handle NaNs and infinities inside data)
vif_df = vif_df.replace([np.inf, -np.inf], np.nan)
vif_df = vif_df.fillna(vif_df.median())

# 3. Handle zero-variance columns (e.g., if every row has the same value)
zero_variance_cols = [col for col in vif_df.columns if vif_df[col].nunique() <= 1]
if zero_variance_cols:
    print(f"⚠️ Removing constant data columns with no variation: {zero_variance_cols}")
    vif_df = vif_df.drop(columns=zero_variance_cols)

# 4. Add the constant column required by statsmodels
vif_df['Constant'] = 1.0

# 5. Calculate VIF and gracefully capture the division-by-zero columns
vif_list = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning) # Mute the warning text in console
    for i in range(vif_df.shape[1]):
        try:
            vif = variance_inflation_factor(vif_df.values, i)
            vif_list.append(vif)
        except ZeroDivisionError:
            vif_list.append(float('inf'))

# 6. Format into a summary table
vif_summary = pd.DataFrame({
    "Feature": vif_df.columns,
    "VIF_Score": vif_list
})

# 7. Remove the 'Constant' tracker and display sorted scores
vif_summary = vif_summary[vif_summary['Feature'] != 'Constant']
vif_summary = vif_summary.sort_values(by="VIF_Score", ascending=False)

print("\n--- Features Ranked by Multicollinearity (VIF) ---")
print(vif_summary.to_string(index=False))


In [ ]:
# 1. Filter your summary table to show only features breaking the threshold
high_vif_df = vif_summary[vif_summary['VIF_Score'] > 10.0].copy()

# 2. Calculate data variance context (Unique values and missing percentages)
unique_counts = []
missing_percentages = []

for col in high_vif_df['Feature']:
    unique_counts.append(employee_featured[col].nunique())
    missing_percentages.append((employee_featured[col].isna().sum() / len(employee_featured)) * 100)

high_vif_df['Unique_Values_Count'] = unique_counts
high_vif_df['Missing_Data_Pct'] = missing_percentages

print("--- Multicollinearity Diagnostic Dashboard ---")
print(high_vif_df.to_string(index=False))

In [ ]:
features_to_drop= ['EngagementCompositeScore', 'CompensationIncomeDifference', 'CompensationDiscrepancy']
employee_featured.drop(columns= features_to_drop, inplace= True)

In [ ]:
employee_featured.columns

In [ ]:
Y= employee_featured['Attrition']
X= employee_featured.drop(columns=['Attrition'], errors= 'ignore')
X_train, x_test, Y_train, y_test= train_test_split(X,Y, test_size= 0.2, stratify= Y, random_state = 0)

categorical_cols= X.select_dtypes(include=['object']).columns.to_list()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

numeric_cols = []
for col in numerical_cols:
    # If the column contains non-numeric values, move it to categorical instead
    if pd.to_numeric(X[col], errors='coerce').isna().all() and len(X[col]) > 0:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)
        


In [ ]:
numeric_cols

In [ ]:
preprocessor= ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

X_train_preprocessed= preprocessor.fit_transform(X_train)
x_test_preprocessed= preprocessor.transform(x_test)

encoded_cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
final_column_names = numeric_cols + encoded_cat_names

In [ ]:
print(Y_train.value_counts())

In [ ]:
smote = ADASYN(random_state=0, sampling_strategy= 0.91)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_preprocessed, Y_train)
X_train_final = pd.DataFrame(X_train_resampled, columns=final_column_names, index=None)
y_train_final = pd.Series(y_train_resampled, name='Attrition')

In [ ]:
X_train_final = pd.DataFrame(X_train_resampled, columns=final_column_names)
Y_train_final = pd.Series(y_train_resampled, name='Attrition')
X_test_final = pd.DataFrame(x_test_preprocessed, columns=final_column_names, index=x_test.index)

In [ ]:
X_train_final.to_csv('X_train_final.csv', index=False)
X_test_final.to_csv('X_test_final.csv', index=True)

Y_train_final.to_csv('y_train_final.csv', index=False)
y_test.to_csv('y_test_final.csv', index=True)

In [ ]:
import joblib
joblib.dump(preprocessor, 'preprocessor_pipeline.pkl')